<h1 style="color:#ffffff; background-color:#2F4538; text-align:center; font-weight:bold; padding:16px 10px; border-radius:8px; margin-bottom:6px;">Day 5 — Scikit-learn Pipelines &amp; Tuned Mini-Project</h1>
<h3 style="color:#3E5C4A; text-align:center; font-weight:bold; margin-top:0;">Chaining Preprocessing and Modeling Into One Leak-Free Object</h3>


<a id="toc"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">Table of Contents</h2>
<div style="border:1px solid #ccc; padding:15px 25px; border-radius:6px;">
<ol style="font-weight:bold; line-height:1.9;">
<li><a href="#section0">0. Setup — Importing Pandas, NumPy &amp; Scikit-learn</a></li>
<li><a href="#section1">1. Why Pipelines Exist</a></li>
<li><a href="#section2">2. Building a Pipeline</a></li>
<li><a href="#section3">3. ColumnTransformer for Mixed Data</a></li>
<li><a href="#section4">4. Tuning a Whole Pipeline</a></li>
<li><a href="#section5">5. The Week 4 Mini-Project</a></li>
<li><a href="#section6">6. Common Mistakes to Avoid</a></li>
<li><a href="#section7">7. Quick Reference</a></li>
<li><a href="#section8">8. Hands-On Lab — Tuned End-to-End Pipeline</a></li>
<li><a href="#section9">9. Best Practices &amp; Reproducibility</a></li>
<li><a href="#section10">10. Summary — What I Learned This Week</a></li>
</ol>
</div>


<a id="section0"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">0. Setup — Importing Pandas, NumPy &amp; Scikit-learn</h2>
<div style="border-left:5px solid #4A9DE0; background-color:#eaf4fc; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1c4e6e;">
<b>Note:</b> Days 1-4 taught splitting, cross-validation, diagnosing fit, engineering features, and tuning hyperparameters — but always as separate manual steps. Today those steps get chained into a single object, so the whole workflow becomes reproducible and structurally safe from leakage.
</div>


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

print("Pandas version      :", pd.__version__)
print("NumPy version       :", np.__version__)
print("Scikit-learn version:", sklearn.__version__)

Pandas version      : 2.3.3
NumPy version       : 2.3.5
Scikit-learn version: 1.7.2


<a id="section1"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">1. Why Pipelines Exist</h2>
<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">1.1 What is data leakage, precisely?</h3>
<b>Data leakage</b> happens whenever information from outside the training set influences training — even in a small, easy-to-miss way. Two very common ways it sneaks in: scaling the <i>whole</i> dataset before splitting (so the scaler has already "seen" the test set's values), or fitting a scaler on validation folds during cross-validation instead of refitting it fresh for each fold.

<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">1.2 Why leakage is dangerous, not just "technically wrong"</h3>
A leaky evaluation quietly inflates your score. The model looks better than it really is during development, then underperforms once it meets genuinely new data in the real world — precisely the situation every evaluation technique this week (three-way splits, cross-validation, held-out test sets) was designed to prevent.

<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">1.3 What a Pipeline is</h3>
A <b>Pipeline</b> chains preprocessing and modeling into <i>one object</i> that applies every step in the correct order, automatically. Instead of you remembering to fit the scaler on training data only, every time, by hand, the Pipeline enforces that discipline structurally.

<div style="border-left:5px solid #3E5C4A; background-color:#eef3ee; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1F2E25;">
<b>Why it matters in AI:</b> Leakage is one of the most common ways an otherwise correct-looking ML project silently fails in production. A model that scored 90% in a notebook but leaked test information during preprocessing can perform far worse once deployed — this is why Pipelines are considered standard, non-optional practice in professional ML code, not just a convenience.
</div>


<a id="section2"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">2. Building a Pipeline</h2>
<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">2.1 What goes inside a Pipeline</h3>
A Pipeline is built from a list of named steps, each one an object with <code>.fit()</code>/<code>.transform()</code> (preprocessing steps) or <code>.fit()</code>/<code>.predict()</code> (the final model step). Every step except the last must be a transformer; the last step is the estimator.

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42)),
])
pipe

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2


<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">2.2 What actually happens when you call .fit() and .predict()</h3>
<code>pipe.fit(X_train, y_train)</code> scales the training data, <i>then</i> trains the model — in one call. <code>pipe.predict(X_test)</code> scales the test data using the <i>same</i> scaler statistics learned from training, then predicts. The scaler is never refit on test or validation data.

<div style="border-left:5px solid #3E5C4A; background-color:#eef3ee; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1F2E25;">
<b>Why it matters in AI:</b> The key benefit: when you call <code>pipe.fit()</code>, the scaler is fit on the training data only; when you cross-validate the pipeline, each fold is scaled using only that fold's training portion. Leakage becomes <i>structurally impossible</i> — not just something you have to remember to avoid by being careful.
</div>


<div style="border-left:5px solid #E85D9A; background-color:#fdeef4; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a2b52;">
<b>Tip:</b> Think of a Pipeline as a single "super-estimator": from the outside, it behaves exactly like any model — it has <code>.fit()</code>, <code>.predict()</code>, and can be cross-validated or grid-searched like any other estimator.
</div>


<a id="section3"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">3. ColumnTransformer for Mixed Data</h2>
<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">3.1 Why one preprocessor isn't enough</h3>
Real datasets have both numeric columns (which usually need scaling) and categorical columns (which usually need encoding). A single <code>StandardScaler</code> cannot sensibly be applied to a text category, and a single <code>OneHotEncoder</code> makes no sense applied to <code>Age</code>.

<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">3.2 What ColumnTransformer does</h3>
<b>ColumnTransformer</b> applies different preprocessing to different column groups — for example, scaling numeric columns while one-hot encoding categorical ones — inside a single transformer that slots directly into a Pipeline as its first step.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_cols = ["Age", "Fare", "sibsp", "Parch"]
categorical_cols = ["Sex", "Pclass"]

pre = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(), categorical_cols),
])
pipe = Pipeline([("pre", pre), ("model", RandomForestClassifier(random_state=42))])
pipe

,steps,"[('pre', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


<div style="border-left:5px solid #3E5C4A; background-color:#eef3ee; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1F2E25;">
<b>Why it matters in AI:</b> ColumnTransformer is what makes Pipelines practical for real, messy, mixed-type data rather than toy all-numeric datasets. Nearly every real-world tabular dataset in AI/ML practice — customer records, medical data, transaction logs — mixes numeric and categorical columns, so this pattern is used constantly outside the classroom, not just in this course.
</div>


<a id="section4"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">4. Tuning a Whole Pipeline</h2>
<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">4.1 The double-underscore naming convention</h3>
GridSearchCV can tune an entire pipeline at once — searching over both preprocessing options and model hyperparameters together. To reach a hyperparameter that lives <i>inside</i> a named step, use <code>stepname__parameter</code> (a double underscore).

In [4]:
from sklearn.model_selection import GridSearchCV

param_grid = {"model__n_estimators": [100, 200],
              "model__max_depth": [5, 10]}
# grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1")
# grid.fit(X_train, y_train)  -- run for real in Section 8
param_grid

{'model__n_estimators': [100, 200], 'model__max_depth': [5, 10]}

<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">4.2 Why tune the whole object instead of the model alone</h3>
This is how professional ML code is structured: one object, cross-validated and tuned end to end, with no leakage anywhere. Every fold created internally by GridSearchCV refits the <i>entire</i> pipeline — including the scaler and encoder — from scratch on that fold's training portion only.

<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> Naming a hyperparameter without its step prefix (e.g. <code>"n_estimators"</code> instead of <code>"model__n_estimators"</code>) will raise an error, because GridSearchCV needs to know <i>which step inside the pipeline</i> the hyperparameter belongs to.
</div>


<a id="section5"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">5. The Week 4 Mini-Project</h2>
Week 4 closes with a single tuned pipeline: EDA-informed feature engineering, a ColumnTransformer for mixed data, a model, all inside one Pipeline, tuned with GridSearchCV, cross-validated, and evaluated once on a held-out test set.

<div style="border-left:5px solid #3E5C4A; background-color:#eef3ee; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1F2E25;">
<b>Why it matters in AI:</b> This is the exact structure the Phase 3 capstone project requires — professional, reproducible, and leak-free. Everything from Day 1 through Day 4 (splits, cross-validation, bias-variance diagnosis, feature engineering, hyperparameter search) converges into this one object today.
</div>


<a id="section6"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">6. Common Mistakes to Avoid</h2>
<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> <ul>
<li><b>Scaling or encoding before splitting</b> — even one call to <code>fit_transform()</code> on the full dataset before <code>train_test_split</code> leaks test information into training.</li>
<li><b>Forgetting the step-name prefix</b> in a pipeline's <code>param_grid</code> (e.g. writing <code>"max_depth"</code> instead of <code>"model__max_depth"</code>).</li>
<li><b>Applying a scaler or encoder outside the Pipeline</b> "just this once" — this quietly reopens the exact leakage risk Pipelines exist to close.</li>
<li><b>Forgetting to include ALL preprocessing inside the Pipeline</b> — if feature engineering happens outside the Pipeline on the full dataset, it can leak just like scaling would.</li>
<li><b>Testing on the held-out test set more than once</b> — even with a Pipeline, the Day 1 rule still holds: the test set is opened exactly once, after every decision is final.</li>
</ul>
</div>


<a id="section7"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">7. Quick Reference</h2>


<div style="border:1px solid #ccc; border-radius:6px; overflow:hidden;">
<table style="width:100%; border-collapse:collapse;">
<tr style="background-color:#3E5C4A; color:#C9D9B0;"><th style="padding:8px; text-align:left;">Task</th><th style="padding:8px; text-align:left;">Code</th></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Import Pipeline</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>from sklearn.pipeline import Pipeline</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Import ColumnTransformer</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>from sklearn.compose import ColumnTransformer</code></td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Build a simple pipeline</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>Pipeline([("scaler", StandardScaler()), ("model", RandomForestClassifier())])</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Preprocess mixed columns</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>ColumnTransformer([("num", StandardScaler(), numeric_cols), ("cat", OneHotEncoder(), categorical_cols)])</code></td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Reach a step's hyperparameter</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>"stepname__parameter"</code> (double underscore)</td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Tune the whole pipeline</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>GridSearchCV(pipe, param_grid, cv=5, scoring="f1")</code></td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Golden rule</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Every preprocessing step lives inside the Pipeline — nothing is fit outside it</td></tr>
</table>
</div>


<a id="section8"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">8. Hands-On Lab — Tuned End-to-End Pipeline</h2>
<div style="border-left:5px solid #9B6FD4; background-color:#f3edfb; padding:10px 15px; margin:10px 0; border-radius:4px; color:#4d3475;">
<b>Goal:</b> Build a Pipeline with a ColumnTransformer, add the Day 4 engineered features, tune the full pipeline with GridSearchCV and 5-fold cross-validation, evaluate the final tuned pipeline once on the held-out test set against a baseline, and note the finished workflow's structure.
</div>


<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">8.0 Loading the dataset</h3>
We reuse <code>train_and_test2.csv</code>, the same Titanic passenger dataset from Days 1–4, predicting whether a passenger survived.

In [5]:
titanic = pd.read_csv("train_and_test2.csv")
titanic = titanic.rename(columns={"2urvived": "Survived"})

titanic.head()

,Passengerid,Age,Fare,Sex,sibsp,zero,zero.1,zero.2,zero.3,zero.4,...,zero.12,zero.13,zero.14,Pclass,zero.15,zero.16,Embarked,zero.17,zero.18,Survived
0,1,22.0,7.2500,0,1,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0
1,2,38.0,71.2833,1,1,0,0,0,0,0,...,0,0,0,1,0,0,0.0,0,0,1
2,3,26.0,7.9250,1,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,1
3,4,35.0,53.1000,1,1,0,0,0,0,0,...,0,0,0,1,0,0,2.0,0,0,1
4,5,35.0,8.0500,0,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0


<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">Step 1 — Build a Pipeline with a ColumnTransformer handling numeric (scaling) and categorical (encoding) columns</h3>


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Sex and Pclass are treated as categorical here even though they are stored as numbers,
# because their numeric codes carry no natural order for the model to exploit
numeric_cols = ["Age", "Fare", "sibsp", "Parch"]
categorical_cols = ["Sex", "Pclass"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

pipe = Pipeline([
    ("pre", preprocessor),
    ("model", RandomForestClassifier(random_state=42)),
])
pipe

,steps,"[('pre', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">Step 2 — Add the engineered features from Day 4 into the workflow</h3>


In [7]:


from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

def add_engineered_features(df):
    df = df.copy()
    df["family_size"] = df["sibsp"] + df["Parch"] + 1
    df["fare_per_person"] = df["Fare"] / df["family_size"]
    df["is_alone"] = (df["family_size"] == 1).astype(int)

    return df

feature_engineer = FunctionTransformer(add_engineered_features,validate=False)



# Define Column Groups AFTER Feature Engineering
numeric_cols = [
    "Age",
    "Fare",
    "sibsp",
    "Parch",
    "family_size",
    "fare_per_person",
]
categorical_cols = [
    "Sex",
    "Pclass",
    "is_alone",
]

# ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),

    ("cat", OneHotEncoder(handle_unknown="ignore"),
     categorical_cols),
])

# Full End-to-End Pipeline
pipe = Pipeline([
    # Step 1: Create Day 4 engineered features
    ("features", feature_engineer),

    # Step 2: Scale numeric + encode categorical features
    ("pre", preprocessor),

    # Step 3: Train the model
    ("model", RandomForestClassifier(random_state=42)),
])


# Use RAW features only
raw_features = [
    "Age",
    "Fare",
    "Sex",
    "sibsp",
    "Parch",
    "Pclass",
]
X = titanic[raw_features]
y = titanic["Survived"]


# Train / Validation / Test Split
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42
)

print(
    "Train:", X_train.shape,
    "Val:", X_val.shape,
    "Test:", X_test.shape
)

pipe
print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)

Train: (785, 6) Val: (262, 6) Test: (262, 6)
Train: (785, 6)  Val: (262, 6)  Test: (262, 6)


<div style="border-left:5px solid #4A9DE0; background-color:#eaf4fc; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1c4e6e;">
<b>Note:</b> Because these engineered features are simple deterministic formulas (sums, ratios) computed row by row, computing them before the split introduces no leakage here — each row's <code>family_size</code> only depends on that row's own <code>sibsp</code>/<code>Parch</code>. Leakage risk applies to statistics computed <i>across</i> rows, like a column mean or a scaler's fitted range.
</div>


<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">Step 3 — Tune the full pipeline with GridSearchCV and 5-fold cross-validation</h3>


In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10, None],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1")
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print(f"Best cross-validated F1: {grid.best_score_:.3f}")

Best params: {'model__max_depth': 10, 'model__n_estimators': 100}
Best cross-validated F1: 0.547


<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">Step 4 — Evaluate the final tuned pipeline once on the held-out test set and report the metric against a baseline</h3>


In [9]:
from sklearn.metrics import f1_score

# The final, tuned pipeline (already refit on the full training set by GridSearchCV)
best_pipeline = grid.best_estimator_
test_predictions = best_pipeline.predict(X_test)
test_f1 = f1_score(y_test, test_predictions)

# Baseline for comparison: an untuned pipeline with default RandomForest settings,
# same preprocessing, evaluated on the SAME held-out test set
baseline_pipeline = Pipeline([
    ("pre", preprocessor),
    ("model", RandomForestClassifier(random_state=42)),
])
baseline_pipeline.fit(X_train, y_train)
baseline_test_f1 = f1_score(y_test, baseline_pipeline.predict(X_test))

print(f"Baseline pipeline -- held-out test F1 : {baseline_test_f1:.3f}")
print(f"Tuned pipeline    -- held-out test F1 : {test_f1:.3f}")
print(f"Improvement                            : {test_f1 - baseline_test_f1:+.3f}")

ValueError: A given column is not a column of the dataframe

<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> This is the <b>only</b> time the test set is touched in this notebook — exactly once, after every tuning decision was already finalized using cross-validation on the training set. This closes the loop that started with the Day 1 rule about the three-way split.
</div>


<h3 style="color:#C9D9B0; font-weight:bold; background-color:#1F2E25; display:inline-block; padding:4px 10px; border-radius:6px;">Step 5 — Note the finished workflow's structure</h3>


In [ ]:
from sklearn import set_config
set_config(display="diagram")
best_pipeline

<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Final Reflection</b> Bundling the scaler, encoder, and model into one Pipeline made the workflow simpler because preprocessing and modeling could be trained and used as a single object. It also made the process safer because during cross-validation, each preprocessing step was fitted only on the training portion of each fold, which helps prevent data leakage.


<a id="section9"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">9. Best Practices &amp; Reproducibility</h2>
<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> <ul>
<li>Put <b>every</b> preprocessing step inside the Pipeline — scaling, encoding, and any row-independent feature engineering — so nothing is ever fit outside of it.</li>
<li>Use <code>ColumnTransformer</code> whenever numeric and categorical columns need different treatment, rather than preprocessing them by hand in separate steps.</li>
<li>Reach hyperparameters inside a pipeline with the <code>stepname__parameter</code> double-underscore convention.</li>
<li>Open the held-out test set <b>exactly once</b>, only after GridSearchCV has already finalized every tuning decision using cross-validation.</li>
<li>Fix <code>random_state=42</code> on every split and every model, for a fully reproducible pipeline end to end.</li>
</ul>
</div>


<a id="section10"></a>
<h2 style="color:#C9D9B0; background-color:#3E5C4A; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">10. Summary — What I Learned This Week</h2>
<div style="border-left:5px solid #E85D9A; background-color:#fdeef4; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a2b52;">
<b>Tip:</b> <ul>
<li>Day 1: a <b>three-way split</b> (train/validation/test) keeps the test set an honest, one-time estimate.</li>
<li>Day 2: <b>k-fold cross-validation</b> replaces one lucky-or-unlucky split with a stable, averaged estimate, and <b>stratified</b> folds protect imbalanced classification data.</li>
<li>Day 3: every model failure is <b>underfitting</b> or <b>overfitting</b>, diagnosed from the train-vs-validation gap, with regularization as the standard cure for overfitting.</li>
<li>Day 4: <b>feature engineering</b> often matters more than model choice, and <b>hyperparameters</b> should be searched systematically with <code>GridSearchCV</code>, not guessed.</li>
<li>Day 5: a <b>Pipeline</b> with a <b>ColumnTransformer</b> chains all of the above into one leak-free, tunable, reproducible object — the exact professional structure the Phase 3 capstone requires.</li>
</ul>
</div>
